# Wildfire-RL — Authoritative Training on Colab (T4 · High-RAM)

Runs **all the heavy training in parallel** on a Colab **T4 GPU**, saving every checkpoint
**directly to Google Drive**. Re-running is cheap: models already in Drive are **skipped**
(resume), so a disconnect never costs you completed work.

**Workflow**
1. Upload the project **zip** (exported *without* `models/`).
2. **Mount Drive** — models (and results/figures) are written straight to `MyDrive/wildfire-rl/`.
3. Install deps (reuses Colab's CUDA PyTorch — torch is **not** reinstalled).
4. Parallel orchestrator: single-agent PPO (Saudi + California × 5 seeds) **and** heuristic+PPO MARL
   scaling (1/3/5/10) — many jobs at once, existing checkpoints skipped.
5. (Optional) evaluate + transfer + figures + gates as a sanity check.
6. Models already live in Drive → just copy `MyDrive/wildfire-rl/models/` into your local `models/`.

> **Scientific guardrail:** PPO is the honest **negative baseline**; heuristic routing is the
> **effective method**. Nothing is tuned to force a PPO "win" (`REMEDIATION_PLAN.md` §1.2 / Phase 16).

---
> **Runtime:** `Runtime → Change runtime type → T4 GPU` + (Pro) `High-RAM`.


## 1 · Confirm the GPU / runtime

In [ ]:
import subprocess, torch, os, multiprocessing as mp
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "no nvidia-smi")
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("CPU cores:", mp.cpu_count(), "| RAM GB:",
      round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1))

## 2 · Upload & unpack the project zip

Pick your `wildfire-rl` export (`.zip`, without `models/`). We unzip it and `cd` into the detected
repo root (folder containing `pyproject.toml`).

In [ ]:
import zipfile, pathlib, os
from google.colab import files

up = files.upload()                        # pick the .zip
ZIP_PATH = pathlib.Path(next(iter(up)))
# --- or from Drive (uncomment after mounting in the next cell) ----------------
# ZIP_PATH = pathlib.Path("/content/drive/MyDrive/wildfire-rl.zip")

DEST = pathlib.Path("/content/project"); DEST.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DEST)

roots = [p.parent for p in DEST.rglob("pyproject.toml")]
assert roots, "pyproject.toml not found in the zip — did you zip the project *contents*?"
REPO = sorted(roots, key=lambda p: len(p.parts))[0]
os.chdir(REPO)
print("Repo root:", REPO)
print("Top-level:", sorted(p.name for p in REPO.iterdir())[:20])

## 3 · Mount Google Drive — models saved here directly

We point the library's path resolver at Drive via env vars (`paths.py` honours
`WILDFIRE_MODELS_DIR` / `WILDFIRE_RESULTS_DIR` / `WILDFIRE_FIGURES_DIR`). Every training subprocess
launched later **inherits** these, so checkpoints are written **straight to Drive** — no copy step,
and they survive disconnects. Change `DRIVE_ROOT` if you want a different folder.

In [ ]:
import os
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT   = Path("/content/drive/MyDrive/wildfire-rl")   # <-- change if you like
MODELS_DRIVE = DRIVE_ROOT / "models"
for sub in ["models", "results", "figures"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

# these persist for the whole kernel; subprocesses inherit them
os.environ["WILDFIRE_MODELS_DIR"]  = str(DRIVE_ROOT / "models")
os.environ["WILDFIRE_RESULTS_DIR"] = str(DRIVE_ROOT / "results")   # comment out to keep results local
os.environ["WILDFIRE_FIGURES_DIR"] = str(DRIVE_ROOT / "figures")   # comment out to keep figures local

print("Models  ->", os.environ["WILDFIRE_MODELS_DIR"])
print("Results ->", os.environ["WILDFIRE_RESULTS_DIR"])
print("Figures ->", os.environ["WILDFIRE_FIGURES_DIR"])

## 4 · Install dependencies

Keep Colab's CUDA PyTorch; install only the remaining pinned deps + the package with `--no-deps`.

In [ ]:
%%bash
set -e
pip -q install \
  "stable-baselines3==2.3.2" "gymnasium==0.29.1" "omegaconf==2.3.0" \
  "seaborn==0.13.2" "tqdm==4.66.4" "scipy==1.13.0" "pandas>=2.0,<3.0" tensorboard
pip -q install -e . --no-deps
python - <<'PY'
import stable_baselines3, gymnasium, omegaconf, wildfire_rl, torch
print("sb3", stable_baselines3.__version__, "| gym", gymnasium.__version__,
      "| wildfire_rl", wildfire_rl.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())
PY

## 5 · Sanity checks (data tensors, imports, 1-step env smoke)

In [ ]:
import numpy as np, sys, subprocess
from wildfire_rl.paths import region_tensor_path, models_dir, results_dir

print("resolved models_dir ->", models_dir())   # should be the Drive path
for rdir in ["saudi_eastern_province", "california"]:
    tp = region_tensor_path(rdir, 32)
    if not tp.exists():
        print("Building missing tensor for", rdir)
        subprocess.run([sys.executable, "scripts/build_tensors.py", "--region", rdir, "--grid", "32"], check=True)
    print(f"{rdir}: state_tensor {np.load(tp).shape}")

crit = region_tensor_path("saudi_eastern_province", 32, "criticality.npy")
print("Saudi criticality:", "OK" if crit.exists() else "MISSING (run scripts/build_criticality.py)")

from wildfire_rl.envs.base import make_env_factory
from wildfire_rl.config import load_config
cfg = load_config("configs/experiment/multiseed.yaml")
env = make_env_factory(state_tensor=np.load(region_tensor_path(cfg.region.dir, 32)), config=cfg.env)()
env.reset(seed=0); env.step(env.action_space.sample())
print("env OK")

## 6 · Experiment knobs

Defaults reproduce the authoritative surface (5 seeds × 100k steps/region). Set `FORCE_RETRAIN=True`
to ignore existing Drive checkpoints; otherwise they are **skipped** (resume).

`N_PARALLEL` = concurrent training jobs. The CNN is tiny → each job is CPU/env-step bound
(GPU ≈ CPU here), so overlapping independent jobs is what shortens wall-clock. All share one T4.

In [ ]:
import multiprocessing as mp

SEEDS            = [0, 1, 2, 3, 4]
TOTAL_TIMESTEPS  = 100_000                  # per model (use 5_000 for a dry run first)
REGIONS          = [("multiseed.yaml", "saudi"), ("multiseed_california.yaml", "california")]

RUN_SINGLE_AGENT = True
RUN_MARL_SCALING = True
MARL_TIMESTEPS   = 100_000

FORCE_RETRAIN    = False                    # True = retrain even if a checkpoint already exists
USE_GPU          = True                     # False = force CPU per worker (spread across cores)
N_PARALLEL       = min(4, mp.cpu_count())

RUN_EVAL = RUN_TRANSFER = RUN_FIGURES = RUN_GATES = True
print(f"{len(SEEDS)*len(REGIONS)} single-agent jobs | N_PARALLEL={N_PARALLEL} | "
      f"{TOTAL_TIMESTEPS:,} steps/model | FORCE_RETRAIN={FORCE_RETRAIN} | "
      f"GPU={USE_GPU and torch.cuda.is_available()}")

## 7 · Parallel training orchestrator (with resume)

Each `(region, seed)` PPO model trains as an **independent subprocess** (own Python/torch/CUDA
context — no GIL contention), `N_PARALLEL` at a time, streaming to its own log. Checkpoints already
present in Drive are **skipped** unless `FORCE_RETRAIN`.

In [ ]:
import subprocess, os, sys, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from wildfire_rl.paths import models_dir

LOGDIR = Path("/content/colab_logs"); LOGDIR.mkdir(parents=True, exist_ok=True)

def ckpt_path(region, s):     # matches cmd_train: ppo_{name}_{grid}_seed_{s}.zip
    return models_dir() / f"ppo_{region}_32_seed_{s}.zip"

def build_jobs():
    jobs, skipped = [], []
    if RUN_SINGLE_AGENT:
        for cfg_name, region in REGIONS:
            for s in SEEDS:
                if ckpt_path(region, s).exists() and not FORCE_RETRAIN:
                    skipped.append(ckpt_path(region, s).name); continue
                jobs.append({"name": f"train_{region}_seed_{s}",
                             "cmd": [sys.executable, "scripts/train.py",
                                     "--config", f"configs/experiment/{cfg_name}",
                                     "--set", f"seeds=[{s}]", f"ppo.total_timesteps={TOTAL_TIMESTEPS}"]})
    if skipped:
        print(f"Resume: skipping {len(skipped)} existing checkpoint(s):", ", ".join(skipped))
    return jobs

def run_job(job):
    env = os.environ.copy()                 # inherits WILDFIRE_*_DIR (Drive)
    env["OMP_NUM_THREADS"] = env["MKL_NUM_THREADS"] = "2"
    if not USE_GPU:
        env["CUDA_VISIBLE_DEVICES"] = ""
    logp = LOGDIR / f"{job['name']}.log"; t0 = time.time()
    with open(logp, "w") as f:
        rc = subprocess.run(job["cmd"], env=env, stdout=f, stderr=subprocess.STDOUT).returncode
    return job["name"], rc, time.time() - t0, logp

jobs = build_jobs()
print(f"Launching {len(jobs)} job(s), {N_PARALLEL} at a time...\n")
results = []
if jobs:
    with ThreadPoolExecutor(max_workers=N_PARALLEL) as ex:
        futs = {ex.submit(run_job, j): j for j in jobs}
        for fut in as_completed(futs):
            name, rc, dt, logp = fut.result()
            print(f"[{'  OK ' if rc == 0 else f'FAIL{rc}'}] {name:26s} {dt:6.0f}s -> {logp}")
            results.append((name, rc))
else:
    print("Nothing to train (all checkpoints present). Set FORCE_RETRAIN=True to redo.")

fails = [n for n, rc in results if rc != 0]
if fails:
    print(f"\n FAILURES: {fails}\n---- tail of {fails[0]}.log ----")
    print((LOGDIR / f"{fails[0]}.log").read_text()[-2000:])
else:
    print("\nSingle-agent training complete — checkpoints saved to Drive:", models_dir())

## 8 · Heuristic + PPO MARL scaling (parallel inside the driver)

Team sizes **1/3/5/10** across NoOp/Random/NearestFire/Frontier/PPO. Skipped if the scaling CSV
already exists (unless `FORCE_RETRAIN`).

In [ ]:
import subprocess, sys, time
from wildfire_rl.paths import results_dir
marl_csv = results_dir() / "marl_scaling_results.csv"
if RUN_MARL_SCALING and (FORCE_RETRAIN or not marl_csv.exists()):
    t0 = time.time()
    p = subprocess.run([sys.executable, "scripts/run_marl_evaluation.py", "--timesteps", str(MARL_TIMESTEPS)],
                       capture_output=True, text=True)
    print(p.stdout[-3000:]); print(p.stderr[-2000:] if p.returncode else "")
    print(f"\nMARL scaling {'OK' if p.returncode==0 else 'FAILED'} in {time.time()-t0:.0f}s")
elif marl_csv.exists():
    print(f"Resume: {marl_csv} exists — skipping. Set FORCE_RETRAIN=True to redo.")
else:
    print("skipped (RUN_MARL_SCALING=False)")

## 9 · Evaluation (heuristics + PPO, both regions) — optional

In [ ]:
import subprocess, sys, glob
import pandas as pd
if RUN_EVAL:
    for cfg_name, region in REGIONS:
        print(f"=== evaluate {region} ===")
        p = subprocess.run([sys.executable, "scripts/evaluate.py", "--config", f"configs/experiment/{cfg_name}"],
                           capture_output=True, text=True)
        print(p.stdout[-1200:]); print(p.stderr[-800:] if p.returncode else "")
    from wildfire_rl.paths import results_dir
    for f in sorted(glob.glob(str(results_dir() / "eval_*.csv"))):
        print("\n#", f); print(pd.read_csv(f)[["policy","burned_cells_mean","reward_mean"]].to_string(index=False))
else:
    print("skipped (RUN_EVAL=False)")

## 10 · Transfer matrix + figures — optional

In [ ]:
import subprocess, sys
if RUN_TRANSFER:
    p = subprocess.run([sys.executable, "scripts/transfer.py", "--config", "configs/experiment/transfer.yaml"],
                       capture_output=True, text=True)
    print("transfer:", "OK" if p.returncode == 0 else "FAILED"); print(p.stdout[-1200:]); print(p.stderr[-1200:] if p.returncode else "")
if RUN_FIGURES:
    p = subprocess.run([sys.executable, "scripts/make_figures.py"], capture_output=True, text=True)
    print("figures:", "OK" if p.returncode == 0 else "FAILED"); print(p.stdout[-1000:])

## 11 · Gates — seed integrity + effective-method gate

`check_seed_integrity.py` → **OK** (distinct checkpoints/rows). `validate_learning_gate.py` is the
**effective-method gate**: best policy (heuristic) beats no-op; PPO printed as a negative result —
that PASS is the intended, honest outcome.

In [ ]:
import subprocess, sys
if RUN_GATES:
    for script in ["scripts/check_seed_integrity.py", "scripts/validate_learning_gate.py"]:
        p = subprocess.run([sys.executable, script], capture_output=True, text=True)
        print(f"=== {script}  (exit {p.returncode}) ===")
        print(p.stdout[-1600:]); print(p.stderr[-800:] if p.stderr else "")
else:
    print("skipped (RUN_GATES=False)")

## 12 · Phase 15B.8 strategic ablations — optional (guarded)

Runs the eight ablation groups **once those scripts land** (`scripts/run_ablations.py`). Flip
`RUN_ABLATIONS_15B8=True` after Phase 15B.8 is implemented; it is a safe no-op until then.

In [ ]:
import subprocess, sys
from pathlib import Path
RUN_ABLATIONS_15B8 = False    # <-- set True once Phase 15B.8 scripts exist
abl = Path("scripts/run_ablations.py")
if RUN_ABLATIONS_15B8 and abl.exists():
    p = subprocess.run([sys.executable, str(abl), "--group", "all"], capture_output=True, text=True)
    print(p.stdout[-3000:]); print(p.stderr[-1500:] if p.returncode else "")
    print("ablations:", "OK" if p.returncode == 0 else "FAILED")
elif RUN_ABLATIONS_15B8:
    print("scripts/run_ablations.py not found — land Phase 15B.8 first.")
else:
    print("15B.8 ablations disabled (set RUN_ABLATIONS_15B8=True once the scripts land).")

## 13 · Confirm models in Drive

Checkpoints are already in Drive (no download needed). Copy `MyDrive/wildfire-rl/models/` into your
local repo's `models/` folder to continue the pipeline. A downloadable zip is offered too.

In [ ]:
import glob, os
from wildfire_rl.paths import models_dir
mroot = models_dir()
models = sorted(glob.glob(str(mroot / "**/*.zip"), recursive=True))
print("Models in Drive:", mroot, "| count:", len(models))
for m in models:
    print("  ", os.path.relpath(m, mroot), round(os.path.getsize(m) / 1e6, 2), "MB")
assert models, "No models found — check the training logs above."

# optional: also grab a single zip to your machine
MAKE_DOWNLOAD_ZIP = False
if MAKE_DOWNLOAD_ZIP:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/models_trained", "zip", str(mroot.parent), mroot.name)
    files.download("/content/models_trained.zip")
print("\nDone. Models persist in Google Drive; copy them into your local models/ for next steps.")